In [1]:
import numpy as np
from numpy import linalg as LA

import matplotlib.pyplot as plt
import time
from joblib import Parallel, delayed
from scipy.integrate import quad
from numpy.random import rand
from scipy.optimize import minimize

import matplotlib as mpl

In [2]:
#high strain limit

#auxiliary matrices for the Jacobian
def B_1(b):
    B=np.zeros((sigma_u,sigma_u))
    for j in range (sigma_u):
        for m in range (sigma_u):
            B[j][m]=np.pi*m*(b[abs(j-m)]+b[j+m])
    return B

def B_2(b):
    B=np.zeros((sigma_u,sigma_u))
    for j in range (sigma_u):
        for m in range (sigma_u):
            B[j][m]=np.pi*m*(b[abs(j-m)]-b[j+m])
    return B

def C_1(c):
    C=np.zeros((sigma_u,sigma_u))
    for j in range (sigma_u):
        for m in range (sigma_u):
            C[j][m]=np.pi*m*(c[abs(j-m)]*np.sign(j-m)-c[j+m])
    return C

def C_2(c):
    C=np.zeros((sigma_u,sigma_u))
    for j in range (sigma_u):
        for m in range (sigma_u):
            C[j][m]=np.pi*m*(c[abs(j-m)]*np.sign(j-m)+c[j+m])
    return C

#Jacobian
def Jac(B1,B2,C1,C2,y,eta,sigma_u):
    vec_aux=np.array([j**2*np.pi**2 for j in range (sigma_u)])
    D=np.diag(vec_aux)
    N_u=sigma_u
    zero_Nu=np.zeros((N_u,N_u))
    ident_Nu=np.identity(N_u)
    return np.block([[zero_Nu,ident_Nu,zero_Nu,zero_Nu],[-y*D-y*C1,-eta*D-eta*C1,-y*B1,-eta*B1],\
                    [zero_Nu,zero_Nu,zero_Nu,ident_Nu],[y*B2,eta*B2,-y*D-y*C2,-y*D-y*C2]])

# Minimization function, maximum Lyapunov exponent
def objective(x,y,eta,sigma_u):
    b,c=x[:2*sigma_u],x[2*sigma_u:]
    B1,B2,C1,C2=B_1(b),B_2(b),C_1(c),C_2(c)
    J=Jac(B1,B2,C1,C2,y,eta,sigma_u)
    EV=np.sort(np.real(LA.eigvals(J)))
    return EV[len(EV)-5] #Exclude the four identically zero eigenvalues


#A'/A as a function of x
def frac_A(x,b,c,sigma_u):
    sum_N=b[0]
    for n in range (1,2*sigma_u):
        sum_N+=2*(b[n]*np.cos(n*np.pi*x)-c[n]*np.sin(n*np.pi*x))
    return sum_N

#Variance of A'/A
def integrand_var(x,b,c,sigma_u):
    mu, _ = quad(frac_A, 0, 1, args=(b,c,sigma_u))
    return (frac_A(x,b,c,sigma_u) - mu)**2

#Run dynamics, return the improvement in the decay rate and the standard deviation
def run_dynamics(trial_y,trial_eta,sigma_u,n_iter,bounds_min,eps,n_y,n_eta,N_A):
    rng = np.random.default_rng() 
    y=2.5+trial_y/(n_y)
    eta=1.5+trial_eta/n_eta
    p0=-eps + rng.random(N_A)*2*eps#Choose a random initial point with uniform density
    result_min = minimize(objective, x0=p0, method='SLSQP',bounds=bounds_min, options={'maxiter':n_iter}, tol=10**(-7), args=(y,eta,sigma_u))
    
    min_fun=(result_min.fun-objective(np.zeros(N_A),y,eta,sigma_u))/objective(np.zeros(N_A),y,eta,sigma_u)#normalized improvement
    min_x=result_min.x
    b,c=min_x[:2*sigma_u],min_x[2*sigma_u:]

    variance, _ = quad(integrand_var, 0, 1, args=(b,c,sigma_u))
    sigma = np.sqrt(variance)
    return min_fun, sigma

In [3]:
start=time.time()

n_p0=10**(1)#number of different initial points
n_y=10**(1)#number of different values for the Young's modulus
n_eta=10**(1)#number of different values for the viscosity
eps=2#bound the possible values for the mode coefficients
n_iter=10**(5)#number of iterations for the minimization algorithm
sigma_u=2#number of modes
N_A=4*sigma_u

bounds_min=np.zeros((N_A,2))
for i in range (N_A):
    bounds_min[i]=[-eps,eps]

min_fun=10
min_x=np.full(N_A,-1)

#Run n_y*n_eta*n_p0 different minimizations
results = Parallel(n_jobs=-1)(delayed(run_dynamics)(trial_y,trial_eta,sigma_u,n_iter,bounds_min,eps,n_y,n_eta,N_A) 
                             for trial_y in range (n_y) for trial_eta in range (n_eta) for trial in range (n_p0))

end=time.time()
#Return total time needed for the computation
print(end-start)

10.332634687423706


In [4]:
#For every value of the pair (y, eta), select the initial condition that led 
#to the best relative decay rate improvement

results_fun=np.zeros(len(results))
results_sigma=np.zeros(len(results))
for i in range (len(results)):
    results_fun[i]=results[i][0]
    results_sigma[i]=results[i][1]

results_fun_tensor=results_fun.reshape(n_y, n_eta, n_p0)
results_sigma_tensor=results_sigma.reshape(n_y, n_eta, n_p0)
results_fun_optimum=-10*np.ones((n_y, n_eta))
results_sigma_optimum=np.zeros((n_y, n_eta))
for i in range (n_y):
    for j in range (n_eta):
        for k in range (n_p0):
            if results_fun_tensor[i][j][k]>results_fun_optimum[i][j]:
                results_fun_optimum[i][j]=results_fun_tensor[i][j][k]
                results_sigma_optimum[i][j]=results_sigma_tensor[i][j][k]

In [20]:
#low strain limit functions

#1/A
def integrand(x,b,c):
    A_tot=0
    for n in range (2*sigma_u-1):
        A_tot+=2*(b[n]*np.cos(n*np.pi*x)-c[n]*np.sin(n*np.pi*x))
    return 1/A_tot

#integral of 1/A(x)
def integral_A(b,c):
    return quad(integrand, 0, 1,args=(b,c))[0]

#B_mat,C_mat
def B_1(b):
    B=np.zeros((N_u,N_u))
    for j in range (1,sigma_u):
        for m in range (1,sigma_u):
            B[j-1][m-1]=b[abs(j-m)]+b[j+m]
    return B

def B_2(b):
    B=np.zeros((N_u,N_u))
    for j in range (1,sigma_u):
        for m in range (1,sigma_u):
            B[j-1][m-1]=b[abs(j-m)]-b[j+m]
    return B

def C_1(c):
    C=np.zeros((N_u,N_u))
    for j in range (1,sigma_u):
        for m in range (1,sigma_u):
            C[j-1][m-1]=-c[abs(j-m)]*np.sign(j-m)+c[j+m]
    return C

def C_2(c):
    C=np.zeros((N_u,N_u))
    for j in range (1,sigma_u):
        for m in range (1,sigma_u):
            C[j-1][m-1]=c[abs(j-m)]*np.sign(j-m)+c[j+m]
    return C

#Jacobian
def Jac(x,y,eta,N_u):
    b,c=x[:2*sigma_u-1],x[2*sigma_u-1:]
    B1,B2,C1,C2=B_1(b),B_2(b),C_1(c),C_2(c)

    I_a=integral_A(b,c)
    vec_aux=np.array([I_a/(j**2*np.pi**2) for j in range (1,sigma_u)])
    D=np.diag(vec_aux)
    zero_Nu=np.zeros((N_u,N_u))
    ident_Nu=np.identity(N_u)
    
    J=np.block([[-eta/y*ident_Nu,-np.matmul(D,B1)/y,zero_Nu,np.matmul(D,C1)/y],[ident_Nu,zero_Nu,zero_Nu,zero_Nu],
                    [zero_Nu,-np.matmul(D,C2)/y,-eta/y*ident_Nu,-np.matmul(D,B2)/y],[zero_Nu,zero_Nu,ident_Nu,zero_Nu]])
    return LA.inv(J)

# Minimization function
def objective(x,y,eta,N_u):
    J=Jac(x,y,eta,N_u)
    EV=np.real(np.sort(LA.eigvals(J)))
    return EV[len(EV)-1]

#A as a function of x
def A_x(x,b,c,sigma_u):
    sum_N=b[0]
    for n in range (1,2*sigma_u-1):
        sum_N+=2*(b[n]*np.cos(n*np.pi*x)-c[n]*np.sin(n*np.pi*x))
    return sum_N

#Compute minimum of the cross-sectional area
def check_pos(min_x):
    b,c=min_x[:2*sigma_u-1],min_x[2*sigma_u-1:]
    n = 100
    x_segment=np.linspace(-1,1,n)
    sum_N=b[0]
    for n in range (1,2*sigma_u-1):
        sum_N+=2*(b[n]*np.cos(n*np.pi*x_segment)-c[n]*np.sin(n*np.pi*x_segment))
    return np.min(sum_N)
    
cons = {'type':'ineq', 'fun': check_pos}#dictionary to be used in the minimization function

#Variance
def integrand_var(x,b,c,sigma_u):
    mu, _ = quad(A_x, 0, 1, args=(b,c,sigma_u))
    return (A_x(x,b,c,sigma_u) - mu)**2

#Run dynamics, return the improvement in the decay rate and the standard deviation
def run_dynamics(trial_y,trial_eta,sigma_u,n_iter,bounds_min,eps,n_y,n_eta,N_A,N_u):
    rng = np.random.default_rng() 
    y=0.5+trial_y/(2*n_y)
    eta=1+trial_eta/n_eta
    p0 = -eps+rng.random(N_A)*2*eps
    p0[0]=1
    result_min = minimize(objective, x0=p0, method='SLSQP',bounds=bounds_min, options={'maxiter':n_iter}, tol=10**(-6)\
                          ,args=(y,eta,N_u), constraints=cons)
    vec_test=np.zeros(N_A)
    vec_test[0]=1

    min_fun=(result_min.fun-objective(vec_test,y,eta,N_u))/objective(vec_test,y,eta,N_u)#normalized improvement
    min_x=result_min.x
    b,c=min_x[:2*sigma_u-1],min_x[2*sigma_u-1:]

    variance, _ = quad(integrand_var, 0, 1, args=(b,c,sigma_u))
    sigma = np.sqrt(variance)

    return min_fun, sigma

In [21]:
start=time.time()

n_p0=10**(1)#number of different initial points
n_y=10**(1)#number of different values for the Young's modulus
n_eta=10**(1)#number of different values for the viscosity
eps=0.2#bound the possible values for the mode coefficients
n_iter=10**(5)#number of iterations for the minimization algorithm
sigma_u=2
N_A=4*sigma_u-2
N_u=sigma_u-1

bounds_min=np.zeros((N_A,2))
for i in range (N_A):
    bounds_min[i]=[-eps,eps]
bounds_min[0]=[0.5,2]


min_fun=10
min_x=np.full(N_A,-1)
#Run n_y*n_eta*n_p0 different minimizations
results = Parallel(n_jobs=-1)(delayed(run_dynamics)(trial_y,trial_eta,sigma_u,n_iter,bounds_min,eps,n_y,n_eta,N_A,N_u) 
                             for trial_y in range (n_y) for trial_eta in range (n_eta) for trial in range (n_p0))
end=time.time()
#Return total time needed for the computation
print(end-start)

58.16539812088013


In [22]:
#For every value of the pair (y, eta), select the initial condition that led 
#to the best relative decay rate improvement

results_fun=np.zeros(len(results))
results_sigma=np.zeros(len(results))
for i in range (len(results)):
    results_fun[i]=results[i][0]
    results_sigma[i]=results[i][1]

results_fun_tensor=results_fun.reshape(n_y, n_eta, n_p0)
results_sigma_tensor=results_sigma.reshape(n_y, n_eta, n_p0)
results_fun_optimum=-10*np.ones((n_y, n_eta))
results_sigma_optimum=np.zeros((n_y, n_eta))
for i in range (n_y):
    for j in range (n_eta):
        for k in range (n_p0):
            if results_fun_tensor[i][j][k]>results_fun_optimum[i][j]:
                results_fun_optimum[i][j]=results_fun_tensor[i][j][k]
                results_sigma_optimum[i][j]=results_sigma_tensor[i][j][k]